# 08 — Registry & Model Resolution

The unified registry is the single source of truth for robot and policy
definitions. JSON-driven, mtime-hot-reloadable, extensible at runtime.

## Architecture

```
registry/
├── robots.json       # 30+ robot definitions (aliases, assets, hardware)
├── policies.json     # Policy providers (module, class, shorthands, url_patterns)
├── loader.py         # JSON loading + mtime cache + _extends validation
├── robots.py         # resolve_name(), get_robot(), list_robots(), has_sim()
├── policies.py       # resolve_policy(), import_policy_class(), build_policy_kwargs()
└── user_registry.py  # register_robot() → ~/.strands_robots/user_robots.json
```

In [ ]:
from strands_robots.registry import (
    resolve_name, get_robot, list_robots, list_aliases,
    has_sim, has_hardware, get_hardware_type,
)

# Alias resolution — normalized lowercase, hyphens → underscores
tests = ["franka", "SO100_follower", "g1", "panda", "so101", "cf2"]
print("alias resolution:")
for alias in tests:
    print(f"  {alias:20s} → {resolve_name(alias)}")

In [ ]:
# List all robots with sim/hardware availability
robots = list_robots()
print(f"\n{len(robots)} robots:")
for name in robots[:12]:
    info = get_robot(name)
    sim = "🎮" if has_sim(name) else "  "
    hw = "🔧" if has_hardware(name) else "  "
    desc = (info.get("description", "") or "")[:45] if info else ""
    print(f"  {sim}{hw} {name:25s} {desc}")
if len(robots) > 12:
    print(f"  ... and {len(robots) - 12} more")

In [ ]:
# Full robot definition
info = get_robot("so100")
if info:
    for k, v in info.items():
        print(f"  {k}: {v}")

## Policy Registry

In [ ]:
from strands_robots.registry import (
    list_policy_providers, get_policy_provider, resolve_policy,
)

print("providers:")
for name in list_policy_providers():
    info = get_policy_provider(name)
    shorthands = info.get("shorthands", []) if info else []
    aliases = info.get("aliases", []) if info else []
    print(f"  {name:20s} shorthands={shorthands} aliases={aliases}")

In [ ]:
# Smart resolution: provider + kwargs from a single string
for s in ["mock", "groot", "lerobot/act_aloha_sim", "zmq://localhost:5555"]:
    try:
        provider, kwargs = resolve_policy(s)
        print(f"  '{s}' → provider='{provider}', kwargs={kwargs}")
    except Exception as e:
        print(f"  '{s}' → {e}")

## Model Resolution (Simulation)

`resolve_model()` finds MJCF/URDF for simulation:
1. Legacy URDF registry (custom `register_urdf()`)
2. Search paths (`STRANDS_ASSETS_DIR`, `~/.strands_robots/assets/`, `./assets/`)
3. Menagerie asset manager (auto-downloads standard robots)

In [ ]:
from strands_robots.simulation.model_registry import (
    resolve_model, list_available_models, register_urdf,
)

# Resolve some robots
for name in ["so100", "panda", "unitree_g1", "crazyflie"]:
    path = resolve_model(name)
    short = str(path)[-60:] if path else "None"
    print(f"  {name:15s} → ...{short}")

## User-Local Registry

Register custom robots that persist across sessions (stored in
`~/.strands_robots/user_robots.json`). Overlays on top of package robots.json.

In [ ]:
from strands_robots.registry import register_robot, list_user_robots, unregister_robot
import tempfile, os

# register_robot requires model_xml — we'll use a temp dir for demo
with tempfile.TemporaryDirectory() as tmpdir:
    # Create a dummy model file
    model_path = os.path.join(tmpdir, "demo_arm.xml")
    with open(model_path, "w") as f:
        f.write("<mujoco><worldbody/></mujoco>")

    register_robot(
        name="demo_arm",
        model_xml="demo_arm.xml",
        description="Demo 3-DOF arm",
        category="arm",
        joints=3,
        asset_dir=tmpdir,
        aliases=["demo"],
    )
    print(f"registered: {list_user_robots()}")

    # Clean up
    unregister_robot("demo_arm")
    print(f"after cleanup: {list_user_robots()}")

## Hot-Reload

The loader uses mtime-based caching. Edit `robots.json` or `policies.json`
and the next query picks up changes automatically — no restart.

```python
from strands_robots.registry import reload
reload()  # force re-read all JSON files
```